## Treinamento da rede e avaliacao para os dados de teste em Python (usando entrada_rede gerada em R)

In [15]:
# Configurações do ambiente em Python

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

SEED = 123
EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 0.0025

np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Carregamento dos dados

entrada_rede = pd.read_csv("../dados/entrada/entrada_rede.csv")

# Split treino/teste
dados_treino = entrada_rede.loc[entrada_rede["conjunto"] == "treino"].copy()
dados_teste = entrada_rede.loc[entrada_rede["conjunto"] == "teste"].copy()

In [17]:
dados_treino.head(20)

,ID,rhc_raca_cor,instrucao,est_cong,rhc_historico_familiar_cancer,rhc_historico_alcool,rhc_historico_tabaco,rhc_origem_encamiamento,rhc_estadiamento_clinico,rhc_primeiro_tratamento_recebido_no_hospital_nenhum,...,macroregiao,pndr_renda,faixa_etcat,ano,tipo_hcido,diff_data_1consulta,diff_data_tratamento,TIME,PSEUDO,SET
0,1,Branca,0,2,Não,NaN,Nunca,Não SUS,NaN,Não,...,3,Média Renda,5,2014,8500,12,42,2.0,1.000005,train
1,1,Branca,0,2,Não,NaN,Nunca,Não SUS,NaN,Não,...,3,Média Renda,5,2014,8500,12,42,8.0,1.000135,train
2,1,Branca,0,2,Não,NaN,Nunca,Não SUS,NaN,Não,...,3,Média Renda,5,2014,8500,12,42,12.0,1.000367,train
3,1,Branca,0,2,Não,NaN,Nunca,Não SUS,NaN,Não,...,3,Média Renda,5,2014,8500,12,42,16.0,-0.039033,train
4,1,Branca,0,2,Não,NaN,Nunca,Não SUS,NaN,Não,...,3,Média Renda,5,2014,8500,12,42,21.1,-0.038593,train
5,1,Branca,0,2,Não,NaN,Nunca,Não SUS,NaN,Não,...,3,Média Renda,5,2014,8500,12,42,28.0,-0.038037,train
6,1,Branca,0,2,Não,NaN,Nunca,Não SUS,NaN,Não,...,3,Média Renda,5,2014,8500,12,42,34.0,-0.037428,train
7,1,Branca,0,2,Não,NaN,Nunca,Não SUS,NaN,Não,...,3,Média Renda,5,2014,8500,12,42,42.0,-0.036837,train
8,1,Branca,0,2,Não,NaN,Nunca,Não SUS,NaN,Não,...,3,Média Renda,5,2014,8500,12,42,53.0,-0.036104,train
9,1,Branca,0,2,Não,NaN,Nunca,Não SUS,NaN,Não,...,3,Média Renda,5,2014,8500,12,42,78.0,-0.035117,train


In [18]:
dados_teste.head(20)

,ID,rhc_raca_cor,instrucao,est_cong,rhc_historico_familiar_cancer,rhc_historico_alcool,rhc_historico_tabaco,rhc_origem_encamiamento,rhc_estadiamento_clinico,rhc_primeiro_tratamento_recebido_no_hospital_nenhum,...,macroregiao,pndr_renda,faixa_etcat,ano,tipo_hcido,diff_data_1consulta,diff_data_tratamento,TIME,PSEUDO,SET
20,3,Não Branca,0,2,NaN,NaN,NaN,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0,2.0,-9.094947e-13,test
21,3,Não Branca,0,2,NaN,NaN,NaN,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0,8.0,-9.094947e-13,test
22,3,Não Branca,0,2,NaN,NaN,NaN,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0,12.0,-1.818989e-12,test
23,3,Não Branca,0,2,NaN,NaN,NaN,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0,16.0,-9.094947e-13,test
24,3,Não Branca,0,2,NaN,NaN,NaN,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0,21.1,-1.818989e-12,test
25,3,Não Branca,0,2,NaN,NaN,NaN,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0,28.0,-2.728484e-12,test
26,3,Não Branca,0,2,NaN,NaN,NaN,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0,34.0,-1.818989e-12,test
27,3,Não Branca,0,2,NaN,NaN,NaN,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0,42.0,-1.818989e-12,test
28,3,Não Branca,0,2,NaN,NaN,NaN,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0,53.0,-9.094947e-13,test
29,3,Não Branca,0,2,NaN,NaN,NaN,SUS,III e IV,Não,...,1,Alta Renda,5,2018,8500,-41,0,78.0,-9.094947e-13,test


In [ ]:
# Carregamento dos dados brutos para obter tempos de evento do teste
cancer_mama = pd.read_csv("../dados/entrada/cancer_mama_bruto.csv")

# IDs únicos do teste e tempos de evento correspondentes
ids_teste = np.sort(dados_teste["id_paciente"].astype(int).unique())
cancer_mama_teste = cancer_mama.loc[cancer_mama["id_paciente"].isin(ids_teste), ["id_paciente", "tempo", "delta_t"]].copy()
tempos_eventos_teste = np.sort(cancer_mama_teste.loc[cancer_mama_teste["delta_t"] == 1, "tempo"].unique())

print(f"Lista tempos de evento (teste): {tempos_eventos_teste}")

Lista tempos de evento (teste): [  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  18  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  46  47  48  49  51  52  55  56  57  60  61  62
  63  67  69  71  73  75  78  81  82  83  84  87  90 112 117]


## Treinamento e Previsões para os dados de teste das Redes Neurais

In [20]:
#Definição da estrutura da rede neural

class rede_sobrevivencia(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.Tanh(),
            nn.Linear(input_dim, 2 * input_dim),
            nn.Tanh(),
            nn.Linear(2 * input_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

## Rede Neural A: one-hot encoding

In [ ]:
def normaliza_coluna_numerica(serie, stats_coluna):
    valores = pd.to_numeric(serie, errors="coerce").fillna(stats_coluna["mediana"])
    return (valores - stats_coluna["media"]) / (stats_coluna["desvio"] + 1e-8)


covariaveis = [col for col in dados_treino.columns if col not in ["id_paciente", "tempo", "pseudo", "conjunto"]]
covariaveis_numericas = [col for col in covariaveis if pd.api.types.is_numeric_dtype(dados_treino[col])]
covariaveis_categoricas = [col for col in covariaveis if col not in covariaveis_numericas]

estatisticas_numericas = {}
for col in covariaveis_numericas:
    serie = pd.to_numeric(dados_treino[col], errors="coerce")
    mediana = float(serie.median()) if serie.notna().any() else 0.0
    serie_imp = serie.fillna(mediana)
    estatisticas_numericas[col] = {
        "mediana": mediana,
        "media": float(serie_imp.mean()),
        "desvio": float(serie_imp.std(ddof=0)),
    }

mapeamento_categorias = {}
for col in covariaveis_categoricas:
    mapeamento_categorias[col] = sorted(dados_treino[col].fillna("NA").astype(str).unique().tolist())


def gera_matriz_design_one_hot(dados, tempos):
    dados_ref = dados.reset_index(drop=True).copy()

    base_num = pd.DataFrame(index=dados_ref.index)
    for col in covariaveis_numericas:
        base_num[col] = normaliza_coluna_numerica(dados_ref[col], estatisticas_numericas[col])

    if len(covariaveis_categoricas) > 0:
        base_cat = dados_ref[covariaveis_categoricas].copy()
        for col in covariaveis_categoricas:
            base_cat[col] = pd.Categorical(
                base_cat[col].fillna("NA").astype(str),
                categories=mapeamento_categorias[col],
                ordered=False,
            )
        base_cat = pd.get_dummies(base_cat, prefix=covariaveis_categoricas, dtype=float)
        base_cat = base_cat.reset_index(drop=True)
    else:
        base_cat = pd.DataFrame(index=dados_ref.index)

    tcat = pd.Categorical(
        pd.to_numeric(dados_ref["tempo"], errors="coerce").astype(float),
        categories=tempos,
        ordered=True,
    )
    t_oh = pd.get_dummies(tcat, prefix="tempo", dtype=float).reset_index(drop=True)

    return pd.concat([base_num, base_cat, t_oh], axis=1)


time_levels = np.sort(dados_treino["tempo"].astype(float).unique())
x_treino_onehot = gera_matriz_design_one_hot(dados_treino, time_levels)
y_treino_onehot = dados_treino["pseudo"].astype(float).values.reshape(-1, 1)

X_train_onehot = torch.tensor(x_treino_onehot.values.astype(np.float32))
y_train_onehot = torch.tensor(y_treino_onehot.astype(np.float32))

train_loader_onehot = DataLoader(
    TensorDataset(X_train_onehot, y_train_onehot),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
 )

print(f"dimensao X_train: {tuple(X_train_onehot.shape)}")
print(f"dimensao y_train: {tuple(y_train_onehot.shape)}")

dimensao X_train: (48610, 63)
dimensao y_train: (48610, 1)


In [22]:
# One hot
x_treino_onehot.head(20)

,instrucao,est_cong,rhc_idade_no_diagnostico_tumor,macroregiao,faixa_etcat,ano,diff_data_1consulta,diff_data_tratamento,rhc_raca_cor_Branca,rhc_raca_cor_NA,...,TIME_2.0,TIME_8.0,TIME_12.0,TIME_16.0,TIME_21.1,TIME_28.0,TIME_34.0,TIME_42.0,TIME_53.0,TIME_78.0
0,-1.583032,0.80868,1.230294,1.744933,1.501794,-0.258050,-0.226377,-0.448011,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-1.583032,0.80868,1.230294,1.744933,1.501794,-0.258050,-0.226377,-0.448011,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-1.583032,0.80868,1.230294,1.744933,1.501794,-0.258050,-0.226377,-0.448011,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-1.583032,0.80868,1.230294,1.744933,1.501794,-0.258050,-0.226377,-0.448011,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-1.583032,0.80868,1.230294,1.744933,1.501794,-0.258050,-0.226377,-0.448011,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
5,-1.583032,0.80868,1.230294,1.744933,1.501794,-0.258050,-0.226377,-0.448011,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
6,-1.583032,0.80868,1.230294,1.744933,1.501794,-0.258050,-0.226377,-0.448011,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
7,-1.583032,0.80868,1.230294,1.744933,1.501794,-0.258050,-0.226377,-0.448011,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
8,-1.583032,0.80868,1.230294,1.744933,1.501794,-0.258050,-0.226377,-0.448011,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
9,-1.583032,0.80868,1.230294,1.744933,1.501794,-0.258050,-0.226377,-0.448011,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [23]:
# Definicao do modelo, criterio e otimizador para a rede one-hot
model_onehot = rede_sobrevivencia(input_dim=X_train_onehot.shape[1])
criterion_onehot = nn.MSELoss()
optimizer_onehot = optim.Adam(model_onehot.parameters(), lr=LEARNING_RATE)

In [24]:
# Treinamento da rede

for epoch in range(EPOCHS):
    model_onehot.train()
    running_loss = 0.0

    for xb, yb in train_loader_onehot:
        optimizer_onehot.zero_grad()
        pred = model_onehot(xb)
        loss = criterion_onehot(pred, yb)
        loss.backward()
        optimizer_onehot.step()
        running_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Época {epoch + 1:4d}/{EPOCHS} | mse={(running_loss / len(train_loader_onehot)):.6f}")

Época   10/100 | mse=0.041513
Época   20/100 | mse=0.023023
Época   30/100 | mse=0.017032
Época   40/100 | mse=0.014109
Época   50/100 | mse=0.012535
Época   60/100 | mse=0.011654
Época   70/100 | mse=0.010906
Época   80/100 | mse=0.010981
Época   90/100 | mse=0.010200
Época  100/100 | mse=0.009756


In [ ]:
# Pasta de saída das predições
dir_saida = Path("../dados/saida")
dir_saida.mkdir(parents=True, exist_ok=True)

# IDs únicos de teste
dados_ids_teste = dados_teste[["id_paciente"] + covariaveis].drop_duplicates().sort_values("id_paciente")

# Base com os tempos de evento do teste para avaliação
linhas_teste_evento = []
for _, linha_individuo in dados_ids_teste.iterrows():
    for tempo_evento in tempos_eventos_teste:
        linha = {
            "id_paciente": int(linha_individuo["id_paciente"]),
            "tempo": float(tempo_evento),
        }
        for col in covariaveis:
            linha[col] = linha_individuo[col]
        linhas_teste_evento.append(linha)
dados_teste_evento = pd.DataFrame(linhas_teste_evento)

# Previsões Rede one-hot
linhas_grade_A = []
for _, linha_individuo in dados_ids_teste.iterrows():
    for tempo_grade in time_levels:
        linha = {
            "id_paciente": int(linha_individuo["id_paciente"]),
            "tempo": float(tempo_grade),
        }
        for col in covariaveis:
            linha[col] = linha_individuo[col]
        linhas_grade_A.append(linha)

grade_predicao_A = pd.DataFrame(linhas_grade_A)
x_pred_A = gera_matriz_design_one_hot(grade_predicao_A, time_levels)
x_pred_A = x_pred_A.reindex(columns=x_treino_onehot.columns, fill_value=0.0)

# Predição da Rede one-hot na grade de treino
model_onehot.eval()
with torch.no_grad():
    grade_predicao_A["pred_s"] = model_onehot(torch.tensor(x_pred_A.values.astype(np.float32))).cpu().numpy().reshape(-1)

grade_predicao_A = grade_predicao_A.sort_values(["id_paciente", "tempo"]).reset_index(drop=True)

# Predição da Rede one-hot nos tempos de evento do teste (via interpolação)
linhas_evento_A = []
for id_individuo, predicoes_individuo in grade_predicao_A.groupby("id_paciente"):
    predicoes_individuo = predicoes_individuo.sort_values("tempo")
    predicoes_evento = np.interp(
        tempos_eventos_teste,
        predicoes_individuo["tempo"].values,
        predicoes_individuo["pred_s"].values,
    )
    for tempo_evento, valor_predito in zip(tempos_eventos_teste, predicoes_evento):
        linhas_evento_A.append({
            "id_paciente": int(id_individuo),
            "tempo": float(tempo_evento),
            "pred_s": float(valor_predito),
        })

pred_event_A = pd.DataFrame(linhas_evento_A).sort_values(["id_paciente", "tempo"]).reset_index(drop=True)

arquivo_saida_A = dir_saida / "predicao-rede-A-onehot.csv"
pred_event_A.to_csv(arquivo_saida_A, index=False)

## Rede Neural B: tempo contínuo

In [ ]:
# Preparação dos dados para a rede usando os tempos originais

def gera_matriz_design_sem_one_hot(dados):
    base = pd.DataFrame(index=dados.index)

    for col in covariaveis_numericas:
        base[col] = normaliza_coluna_numerica(dados[col], estatisticas_numericas[col])

    for col in covariaveis_categoricas:
        categorias = mapeamento_categorias[col]
        serie = dados[col].fillna("NA").astype(str)
        base[col] = pd.Categorical(serie, categories=categorias, ordered=False).codes.astype(float)

    base["tempo"] = pd.to_numeric(dados["tempo"], errors="coerce").astype(float)
    return base

x_treino_B = gera_matriz_design_sem_one_hot(dados_treino)
y_treino_B = dados_treino["pseudo"].astype(float).values.reshape(-1, 1)

X_train_B = torch.tensor(x_treino_B.values.astype(np.float32))
y_train_B = torch.tensor(y_treino_B.astype(np.float32))

train_loader_B = DataLoader(
    TensorDataset(X_train_B, y_train_B),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
 )

In [29]:
# Definição do modelo, critério e otimizador para a rede com tempos originais

model_B = rede_sobrevivencia(input_dim=X_train_B.shape[1])
criterion_B = nn.MSELoss()
optimizer_B = optim.Adam(model_B.parameters(), lr=LEARNING_RATE)

In [30]:
# Treinamento da rede B com tempos originais

for epoch_B in range(EPOCHS):
    model_B.train()
    running_loss_B = 0.0

    for x_batch_B, y_batch_B in train_loader_B:
        optimizer_B.zero_grad()
        pred_B = model_B(x_batch_B)
        loss_B = criterion_B(pred_B, y_batch_B)
        loss_B.backward()
        optimizer_B.step()
        running_loss_B += loss_B.item()

    if (epoch_B + 1) % 10 == 0:
        print(f"Época {epoch_B + 1:4d}/{EPOCHS} | mse_B={(running_loss_B / len(train_loader_B)):.6f}")

Época   10/100 | mse_B=0.060972
Época   20/100 | mse_B=0.057247
Época   30/100 | mse_B=0.054645
Época   40/100 | mse_B=0.053512
Época   50/100 | mse_B=0.052379
Época   60/100 | mse_B=0.051644
Época   70/100 | mse_B=0.050782
Época   80/100 | mse_B=0.050183
Época   90/100 | mse_B=0.049581
Época  100/100 | mse_B=0.049031


In [ ]:
# Predição da Rede B nos tempos de evento do teste
x_evento_B = gera_matriz_design_sem_one_hot(dados_teste_evento)

model_B.eval()
with torch.no_grad():
    pred_s_B = model_B(torch.tensor(x_evento_B.values.astype(np.float32))).cpu().numpy().reshape(-1)

pred_event_B = dados_teste_evento[["id_paciente", "tempo"]].copy()
pred_event_B["pred_s"] = pred_s_B
pred_event_B = pred_event_B.sort_values(["id_paciente", "tempo"]).reset_index(drop=True)

arquivo_saida_B = dir_saida / "predicao-rede-B-tempo-continuo.csv"
pred_event_B.to_csv(arquivo_saida_B, index=False)

### Definição de funções para cálculo do Time Dependent Concordance Index de Antolini

In [ ]:
# Função para calcular o CTD
def calc_ctd(outcomes_df, pred_event_df):

    surv_at_event = pred_event_df.pivot(index="id_paciente", columns="tempo", values="pred_s")
    outcomes = outcomes_df[["id_paciente", "tempo", "delta_t"]].drop_duplicates("id_paciente").copy()
    outcomes = outcomes.sort_values("tempo").reset_index(drop=True)

    concordante = 0
    empates = 0
    pares_comparaveis = 0

    for i in range(len(outcomes)):
        if int(outcomes.loc[i, "delta_t"]) != 1:
            continue

        id_individuo_i = int(outcomes.loc[i, "id_paciente"])
        tempo_individuo_i = float(outcomes.loc[i, "tempo"])

        if id_individuo_i not in surv_at_event.index or tempo_individuo_i not in surv_at_event.columns:
            print(f"Indivíduo {id_individuo_i} não encontrado nas predições, pulando comparação.")
            continue

        risco_at_evento_i = 1.0 - float(surv_at_event.loc[id_individuo_i, tempo_individuo_i])
        js_comparaveis = outcomes.index[outcomes["tempo"] > tempo_individuo_i].tolist()

        for j in js_comparaveis:
            id_individuo_j = int(outcomes.loc[j, "id_paciente"])
            if id_individuo_j not in surv_at_event.index:
                print(f"Indivíduo {id_individuo_j} não encontrado nas predições, pulando comparação.")
                continue

            risco_at_evento_j = 1.0 - float(surv_at_event.loc[id_individuo_j, tempo_individuo_i])
            if np.isnan(risco_at_evento_i) or np.isnan(risco_at_evento_j):
                print(f"Risco NaN para indivíduos {id_individuo_i} ou {id_individuo_j}, pulando comparação.")
                print(f"Risco i: {risco_at_evento_i}, risco j: {risco_at_evento_j}")
                continue

            pares_comparaveis += 1
            if risco_at_evento_i > risco_at_evento_j:
                concordante += 1
            elif risco_at_evento_i == risco_at_evento_j:
                empates += 1

    ctd = (concordante + 0.5 * empates) / pares_comparaveis if pares_comparaveis > 0 else np.nan
    return ctd, pares_comparaveis, concordante, empates

# Monta DF sem extrapolação dos limites de tempo em que a rede foi treinada
def filtrar_dados_sem_extrapolacao(outcomes_df , pred_event_df, time_levels):
    t_min = float(np.min(time_levels))
    t_max = float(np.max(time_levels))

    # Filtra os dados reais (eventos)
    out_test_sem_extrap = outcomes_df.loc[
        (outcomes_df["tempo"] >= t_min) & (outcomes_df["tempo"] <= t_max)
    ].copy()

    # Filtra as predições
    pred_event_sem_extrap = pred_event_df.loc[
        (pred_event_df["tempo"] >= t_min) & (pred_event_df["tempo"] <= t_max)
    ].copy()

    return out_test_sem_extrap, pred_event_sem_extrap


# Resultados Obtidos

### Rede Neural A: one-hot encoding

In [27]:
# Utiliza as predições da rede one-hot
pred_event_A

out_test_sem_extrapA, pred_event_sem_extrapA = filtrar_dados_sem_extrapolacao(
    cancer_mama_teste,
    pred_event_A,
    time_levels)


ctd_sem_extrapA, comp_sem_extrapA, conc_sem_extrapA, emaptes_sem_extrapA = calc_ctd(
    out_test_sem_extrapA,
    pred_event_sem_extrapA,
)

# CTD com todos os eventos (inclui extrapolação)
ctd_todosA, comp_todosA, conc_todosA, ties_todosA = calc_ctd(cancer_mama_teste, pred_event_A)

avaliacao_ctd = pd.DataFrame([
    {
        "CENARIO": "sem_extrapolacao",
        "CTD_TESTE": ctd_sem_extrapA,
        "COMPARAVEIS": comp_sem_extrapA,
        "CONCORDANTES": conc_sem_extrapA,
        "EMPATES": emaptes_sem_extrapA,
    },
    {
        "CENARIO": "todos_eventos",
        "CTD_TESTE": ctd_todosA,
        "COMPARAVEIS": comp_todosA,
        "CONCORDANTES": conc_todosA,
        "EMPATES": ties_todosA,
    },
])

print(avaliacao_ctd)

            CENARIO  CTD_TESTE  COMPARAVEIS  CONCORDANTES  EMPATES
0  sem_extrapolacao   0.592881        76765         34900    21225
1     todos_eventos   0.597877       129315         59451    35727


### Rede Neural B: tempos originais contínuos

In [32]:
# Utiliza as predições da Rede B - tempos originais
pred_event_B

out_test_sem_extrapB, pred_event_sem_extrapB = filtrar_dados_sem_extrapolacao(
    cancer_mama_teste,
    pred_event_B,
    time_levels)


ctd_sem_extrapB, comp_sem_extrapB, conc_sem_extrapB, emaptes_sem_extrapB = calc_ctd(
    out_test_sem_extrapB,
    pred_event_sem_extrapB,
)

# CTD com todos os eventos (inclui extrapolação)
ctd_todosB, comp_todosB, conc_todosB, ties_todosB = calc_ctd(cancer_mama_teste, pred_event_B)

avaliacao_ctd_B = pd.DataFrame([
    {
        "CENARIO": "sem_extrapolacao",
        "CTD_TESTE": ctd_sem_extrapB,
        "COMPARAVEIS": comp_sem_extrapB,
        "CONCORDANTES": conc_sem_extrapB,
        "EMPATES": emaptes_sem_extrapB,
    },
    {
        "CENARIO": "todos_eventos",
        "CTD_TESTE": ctd_todosB,
        "COMPARAVEIS": comp_todosB,
        "CONCORDANTES": conc_todosB,
        "EMPATES": ties_todosB,
    },
])

print(avaliacao_ctd_B)

            CENARIO  CTD_TESTE  COMPARAVEIS  CONCORDANTES  EMPATES
0  sem_extrapolacao   0.618759        76765         47499        0
1     todos_eventos   0.595356       129315         76988        1
